In [ ]:
%pip install torch transformers

In [1]:
from transformers import BertTokenizer
from transformers import BertForSequenceClassification
from transformers import Trainer, TrainingArguments

import sys
import os

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from data.DataLoader import DataLoader
from data.Elastic import Elastic

In [2]:
dataloader = DataLoader(
    Elastic(timeout=5), 
    index='featext'
)

                                           url  type  content_redirects  \
0             http://www.descubrecartagena.com     0                0.0   
1                   http://www.happycheftv.com     0                1.0   
2     http://trade-iq-option-2021.blogspot.com     0                2.0   
3             http://www.http.freecontent.date     0                0.0   
4                     https://draddywe.web.app     0                0.0   
...                                        ...   ...                ...   
3927           http://bt-109938.weeblysite.com     0                0.0   
3928           http://bt-109938.weeblysite.com     0                0.0   
3929           http://bt-109938.weeblysite.com     1                0.0   
3930           http://bt-109938.weeblysite.com     1                0.0   
3931           http://bt-109938.weeblysite.com     1                0.0   

      content_len_html  content_len_text  content_len_links  \
0               1116.0              

In [14]:
duplicated_urls_count = dataloader.df['url'].duplicated().sum()
print(f"Number of duplicated URLs: {duplicated_urls_count}")

dataloader.df.drop_duplicates(subset='url', inplace=True)

Number of duplicated URLs: 26218


In [18]:
print(f"Number of null DOM screenshots: {dataloader.df['dom_screenshot_url'].isna().sum()}")
dataloader.df.dropna(subset=['dom_screenshot_url'], how='all', inplace=True)

Number of null DOM screenshots: 14900


In [21]:
dataloader.df.iloc[1]['dom_screenshot_url']

'https://urlscan.io/screenshots/d1f70d56-7842-4356-87e7-0bca8b1d4556.png'

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch

In [ ]:
urls = dataloader.df['url'].tolist()
labels = dataloader.df['type'].tolist()

In [ ]:
len(urls)

In [ ]:
# Using a pre-trained tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenize URLs
def tokenize_urls(urls):
    return tokenizer(urls, padding=True, truncation=True, max_length=512, return_tensors='pt')

class PhishingDataset(Dataset):
    def __init__(self, encodings, labels):
        # Encodings are expected to be a dict where values are already tensors
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        # Directly use the tensor slices without re-wrapping them into new tensors
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

    def __len__(self):
        return self.labels.size(0)

# Assume URLs and labels are already defined above
encoded_inputs = tokenize_urls(urls)

# Convert the labels to a tensor outside of the dataset initialization to ensure it is properly managed
labels_tensor = torch.tensor(labels, dtype=torch.long)

# Create the dataset
dataset = PhishingDataset(encoded_inputs, labels_tensor)

In [ ]:
# Load pre-trained BERT model for sequence classification
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Freeze all layers except the classifier to speed up training (optional)
for name, param in model.named_parameters():
    if 'classifier' not in name:  # Freeze layers other than the classifier
        param.requires_grad = False

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',          # output directory
    num_train_epochs=3,              # number of training epochs
    per_device_train_batch_size=8,   # batch size for training
    per_device_eval_batch_size=16,   # batch size for evaluation
    warmup_steps=500,                # number of warmup steps for learning rate scheduler
    weight_decay=0.01,               # strength of weight decay
    logging_dir='./logs',            # directory for storing logs
    logging_steps=10,
)

trainer = Trainer(
    model=model,                         # the instantiated Transformers model to be trained
    args=training_args,                  # training arguments, defined above
    train_dataset=dataset,         # training dataset
)

trainer.train()

In [ ]:
trainer.evaluate()